# 00 — Generate Test Data (SCAFFOLDING — not part of the production pipeline)

**What this notebook is:** a stand-in for a real ERP extract. It runs `generate_data.py`, which creates synthetic raw data matching the defect profile described in `schema.yaml`.

**Why it exists:** this capstone has no real Plant System A / Plant System B feed to connect to. `generate_data.py` manufactures one, seeded and reviewed for circularity (Step 1, decision log D-013 to D-017).

**What happens to it in production:** it is deleted. A real deployment replaces this notebook entirely with an actual extract job against the two source systems. Nothing downstream depends on *how* `data_primary/raw/` and `data_control/raw/` were populated — only that they match `schema.raw.*`.

**Output of this notebook:** two folders of raw CSVs — `data_primary/raw/` (realistic ~6% censoring) and `data_control/raw/` (~1.6% censoring control, see decision log D-017). Nothing here is cleaned or validated for use — that's notebook 01.

## Setup — clone the repo fresh

In [ ]:
import subprocess, os

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-1500:])
    return r

os.chdir('/content')
sh('rm -rf ibp-tradeoff')
sh('git clone https://github.com/rdelolmog-creator/ibp-tradeoff.git')
os.chdir('/content/ibp-tradeoff')
print('Working directory:', os.getcwd())


## Generate the primary dataset
Expect: net sales EUR 196,904,149 · conversion cost 0.127 PASS · roll-forward 0.000000 PASS

In [ ]:
sh('python generate_data.py')
sh('mv data data_primary')


## Generate the control dataset (low censoring)
Same script, swapped config, no code edits — this is the reusability evidence for the generator itself.
Expect: net sales EUR 199,190,133 · conversion cost 0.124 PASS

In [ ]:
sh('cp config/assumptions.yaml config/_primary_backup.yaml')
sh('cp config/assumptions_lowcensoring.yaml config/assumptions.yaml')
sh('python generate_data.py')
sh('cp config/_primary_backup.yaml config/assumptions.yaml')
sh('mv data data_control')

print('folders now:', sorted(d for d in os.listdir('.') if os.path.isdir(d)))


## Check: datasets differ only in censoring rate
Expect: `data_primary` ~6.16% · `data_control` ~1.57%

In [ ]:
import pandas as pd
for tag in ['data_primary', 'data_control']:
    a = pd.read_csv(f'{tag}/raw/plant_system_A.csv')
    b = pd.read_csv(f'{tag}/raw/plant_system_B.csv')
    cen = (pd.concat([a.STOCK_CLOSE, b.stock_eom]) <= 0).mean()
    print(f'{tag}: censoring {cen:.2%}')

print()
print('Raw data generation complete.')
print('Next: run 01_ingest_and_clean.ipynb — the actual production pipeline —')
print('against these two raw folders.')


## (Optional) Push the generated raw data to GitHub
Uncomment if you want the raw data visible in the repo for reproducibility, not just re-derivable from the seed. ~280KB total. Not required — the whole point of a seeded generator is that anyone can regenerate identical data without this.

In [ ]:
# sh('git config --global user.email "your@email.com"')
# sh('git config --global user.name "rdelolmog-creator"')
# sh('git add data_primary/raw data_control/raw')
# sh('git commit -m "Add generated raw test data"')
# sh('git push')
